# Hour 2: Function Calling & Tools
**Adding Superpowers to Your Assistant**

## What We're Building:
1. **Calculator Tool** - Accurate math
2. **Web Search Tool** - Current information (mock)
3. **Data Analyzer Tool** - Process CSV/JSON



## Setup

In [1]:
from openai import OpenAI
import os

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✅ Setup complete!")

✅ Setup complete!


---
# Part 1: Calculator Tool

## 🎯 SIMPLE VERSION - Just the Function

In [3]:
# SIMPLE calculator - just 5 lines!
import json

def calculate(expression):
    """Evaluate a math expression"""
    result = eval(expression)
    return json.dumps({"result": result})

# Test it
print(calculate("123 * 456"))
print(calculate("15.5 + 3.2"))

{"result": 56088}
{"result": 18.7}


## 🎯 SIMPLE VERSION - Tool Schema (Tell OpenAI About It)

In [4]:
# Tool schema - describes our function to OpenAI

calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Do accurate math calculations",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Math expression like '2+2' or '10*5'"
                }
            },
            "required": ["expression"]
        }
    }
}

print("✅ Calculator tool defined!")

✅ Calculator tool defined!


## 🎯 SIMPLE VERSION - Use the Tool

In [5]:
# Use calculator with AI - SIMPLE version

# Step 1: Ask AI a math question
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "What's 123 * 456?"}],
    tools=[calculator_tool]  # ← Give AI the tool!
)

# Step 2: Check if AI wants to use the tool
if response.choices[0].message.tool_calls:
    tool_call = response.choices[0].message.tool_calls[0]
    
    # Step 3: Get the arguments AI wants to pass
    args = json.loads(tool_call.function.arguments)
    print(f"AI wants to calculate: {args['expression']}")
    
    # Step 4: Actually run our function
    result = calculate(args['expression'])
    print(f"Result: {result}")
else:
    print("AI didn't use the tool")

AI wants to calculate: 123*456
Result: {"result": 56088}


## 📚 COMPLETE VERSION - With Error Handling

In [6]:
def calculate_safe(expression):
    """
    COMPLETE calculator with error handling.
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return json.dumps({"success": True, "result": result})
    except Exception as e:
        return json.dumps({"success": False, "error": str(e)})

# Test
print(calculate_safe("123 * 456"))
print(calculate_safe("invalid"))  # Handles errors!

{"success": true, "result": 56088}
{"success": false, "error": "name 'invalid' is not defined"}


---
# Part 2: Web Search Tool (Mock)

## 🎯 SIMPLE VERSION

In [7]:
# SIMPLE web search - mock version

def web_search(query):
    """Mock web search - returns pre-defined results"""
    results = {
        "weather": "Sunny, 25°C",
        "news": "AI technology advancing rapidly",
        "stocks": "Tech stocks up 2.5% today"
    }
    
    for keyword in results:
        if keyword in query.lower():
            return json.dumps({"results": results[keyword]})
    
    return json.dumps({"results": "No results found"})

# Test
print(web_search("weather"))
print(web_search("latest news"))

{"results": "Sunny, 25\u00b0C"}
{"results": "AI technology advancing rapidly"}


In [9]:
# Tool schema for web search

web_search_tool = {
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Search for current information",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"}
            },
            "required": ["query"]
        }
    }
}

print("✅ Web search tool defined!")

✅ Web search tool defined!


---
# Part 3: Data Analyzer Tool

## 🎯 SIMPLE VERSION

In [10]:
# SIMPLE data analyzer

def analyze_data(data_string, operation):
    """Analyze a list of numbers"""
    data = json.loads(data_string)
    
    if operation == "sum":
        result = sum(data)
    elif operation == "average":
        result = sum(data) / len(data)
    elif operation == "max":
        result = max(data)
    elif operation == "min":
        result = min(data)
    else:
        result = None
    
    return json.dumps({"result": result})

# Test
print(analyze_data('[10, 20, 30, 40, 50]', 'average'))
print(analyze_data('[10, 20, 30, 40, 50]', 'max'))

{"result": 30.0}
{"result": 50}


In [11]:
# Tool schema for data analyzer

data_analyzer_tool = {
    "type": "function",
    "function": {
        "name": "analyze_data",
        "description": "Analyze numeric data",
        "parameters": {
            "type": "object",
            "properties": {
                "data_string": {"type": "string", "description": "JSON list of numbers"},
                "operation": {
                    "type": "string",
                    "enum": ["sum", "average", "max", "min"],
                    "description": "What to calculate"
                }
            },
            "required": ["data_string", "operation"]
        }
    }
}

print("✅ Data analyzer tool defined!")

✅ Data analyzer tool defined!


---
# Part 4: Using Multiple Tools Together

## 🎯 SIMPLE VERSION - All Tools at Once

In [12]:
# Register all tools
all_tools = [calculator_tool, web_search_tool, data_analyzer_tool]

# Map function names to actual functions
functions = {
    "calculate": calculate,
    "web_search": web_search,
    "analyze_data": analyze_data
}

print("✅ All tools registered!")

✅ All tools registered!


In [13]:
# SIMPLE multi-tool assistant

def ask(question):
    """Ask AI a question - it will use tools automatically!"""
    
    # Step 1: Ask AI
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": question}],
        tools=all_tools
    )
    
    message = response.choices[0].message
    
    # Step 2: Check if AI wants to use tools
    if not message.tool_calls:
        return message.content
    
    # Step 3: Execute tools AI wants
    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        
        print(f"🔧 Using {function_name}...")
        result = functions[function_name](**arguments)
        print(f"   Result: {result}\n")
    
    return "✅ Tools executed!"

# Test it!
ask("What's 15% of 200?")

🔧 Using calculate...
   Result: {"result": 30.0}



'✅ Tools executed!'

## 📚 COMPLETE VERSION - Full Conversation with Tools

In [14]:
def run_conversation(user_message):
    """
    COMPLETE version - runs full conversation with tool execution.
    Returns AI's final answer after using tools.
    """
    messages = [{"role": "user", "content": user_message}]
    
    # First API call
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        tools=all_tools
    )
    
    response_message = response.choices[0].message
    
    # If no tool calls, return text response
    if not response_message.tool_calls:
        return response_message.content
    
    # Add AI's response to conversation
    messages.append(response_message)
    
    # Execute each tool
    for tool_call in response_message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)
        
        print(f"🔧 Calling {function_name} with {function_args}")
        
        # Call the actual function
        function_response = functions[function_name](**function_args)
        
        # Add result to conversation
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": function_response
        })
    
    # Get final answer from AI
    final_response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages
    )
    
    return final_response.choices[0].message.content

# Test it!
print("\nTest 1:")
answer = run_conversation("What's 25% of 80?")
print(f"\nAI: {answer}\n")

print("=" * 70)

print("\nTest 2:")
answer = run_conversation("Find the average of 10, 20, 30, 40, 50")
print(f"\nAI: {answer}")


Test 1:
🔧 Calling calculate with {'expression': '80*0.25'}

AI: 25% of 80 is 20.


Test 2:
🔧 Calling analyze_data with {'data_string': '[10,20,30,40,50]', 'operation': 'average'}

AI: The average of 10, 20, 30, 40, and 50 is 30.


---
# Part 5: Complete Multi-Tool Assistant Class

## 📚 COMPLETE VERSION

In [15]:
class MultiToolAssistant:
    """
    COMPLETE assistant with all tools.
    Tracks usage and handles errors.
    """
    
    def __init__(self):
        self.tools = [calculator_tool, web_search_tool, data_analyzer_tool]
        self.functions = {
            "calculate": calculate_safe,
            "web_search": web_search,
            "analyze_data": analyze_data
        }
        self.tool_usage = {name: 0 for name in self.functions.keys()}
    
    def ask(self, question):
        """Ask a question and get answer (using tools if needed)"""
        print(f"❓ Question: {question}\n")
        
        messages = [{"role": "user", "content": question}]
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages,
            tools=self.tools
        )
        
        response_message = response.choices[0].message
        
        if not response_message.tool_calls:
            return response_message.content
        
        messages.append(response_message)
        
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            # Track usage
            self.tool_usage[function_name] += 1
            
            print(f"🔧 Using {function_name}...")
            
            result = self.functions[function_name](**function_args)
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
        
        final_response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages
        )
        
        print()
        return final_response.choices[0].message.content
    
    def stats(self):
        """Show usage statistics"""
        print("\n📊 Tool Usage Statistics:")
        print("=" * 40)
        for tool, count in self.tool_usage.items():
            print(f"  {tool}: {count} calls")
        print("=" * 40)

# Create assistant
assistant = MultiToolAssistant()
print("✅ Multi-tool assistant ready!\n")

# Test it!
tests = [
    "What's 123 * 456?",
    "What's the weather like?",
    "Find the average of 5, 10, 15, 20, 25"
]

for test in tests:
    print("=" * 70)
    answer = assistant.ask(test)
    print(f"💬 Answer: {answer}\n")

# Show stats
assistant.stats()

✅ Multi-tool assistant ready!

❓ Question: What's 123 * 456?

🔧 Using calculate...

💬 Answer: 123 * 456 equals 56,088.

❓ Question: What's the weather like?

🔧 Using web_search...

💬 Answer: The current weather is sunny with a temperature of 25°C.

❓ Question: Find the average of 5, 10, 15, 20, 25

🔧 Using analyze_data...

💬 Answer: The average of 5, 10, 15, 20, and 25 is 15.0.


📊 Tool Usage Statistics:
  calculate: 1 calls
  web_search: 1 calls
  analyze_data: 1 calls


---
# Summary

## ✅ What You Built:

1. **Calculator Tool** - Accurate math
2. **Web Search Tool** - Current info (mock)
3. **Data Analyzer Tool** - Process data
4. **Multi-Tool Assistant** - Uses all tools automatically

## 🎯 Key Concepts:

1. **Tools = Python functions** with schemas
2. **AI decides** when to use tools
3. **We execute** the actual functions
4. **AI uses results** to answer user

